# Aplicação de *Rapidly-exploring Random Trees* (RRT) na Estação de Hidrogênio (HRS)
### Síntese de Trajetórias Seguras de Abastecimento (SAE J2601) — Setores 200/300 (`RESF-01` & `DISP-01`)

**Disciplina:** ECAA08 • Engenharia de Controle e Automação & Matemática Discreta (UNIFEI)  
**Projeto:** Estação de Reabastecimento de Hidrogênio — `SCADA-Core`

---
### 1. Objetivo da Simulação
Planejar uma trajetória de abastecimento rápido de um veículo FCEV de $50\text{ bar}$ ($25^\circ\text{C}$) até $700\text{ bar}$ utilizando **Kinodynamic RRT** com:
- Modelo termodinâmico simplificado de compressão + efeito Joule-Thomson ($\mu_{\text{JT}} < 0$) + resfriamento via chiller criogênico `RESF-01`;
- Barreira de segurança **SIS / SAE J2601** ($T_{\text{veic}} < 85^\circ\text{C}$, $T_{\text{veic}} \ge -40^\circ\text{C}$, $P_{\text{veic}} < 875\text{ bar}$, `PT-201` $< 900\text{ bar}$);
- Comparação entre a trajetória sintetizada pelo RRT e uma manobra direta em malha aberta sem pré-resfriamento coordenado.

In [ ]:
import time
from typing import List, Tuple, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
np.random.seed(42)
print("Bibliotecas carregadas com sucesso.")

### 2. Modelagem Híbrida do Processo e Barreira Lógica ($\Phi_{\text{SIS}}$)

Definimos o vetor contínuo $\mathbf{x} = [P_{\text{veic}},\; T_{\text{veic}},\; P_{\text{linha}},\; T_{\text{chiller}}]^T$ e o modo discreto $q = (\text{XV}_{101},\; \text{XV}_{201},\; \text{RESF}_{01})$.

In [ ]:
def avaliar_sis_booleano(x: np.ndarray, at101_lfl: float = 2.0, esd100: int = 0) -> Tuple[bool, str]:
    """
    Avalia a barreira logica de seguranca Phi_SIS segundo SAE J2601 e IEC 61511.
    Retorna (seguro: bool, motivo: str).
    """
    p_veic, t_veic, p_linha, t_chiller = x

    if esd100 == 1:
        return False, "ESD-100 Acionado"
    if at101_lfl >= 40.0:
        return False, "AT-101 Vazamento >= 40% LFL"
    if p_veic >= 875.0:
        return False, "Sobrepressao Veiculo >= 875 bar"
    if t_veic >= 85.0:
        return False, "Sobretemperatura Tanque FCEV >= 85 C"
    if t_veic < -40.0 or t_chiller < -45.0:
        return False, "Subtemperatura Critica < -40 C"
    if p_linha >= 900.0:
        return False, "PT-201 Sobrepressao Linha >= 900 bar"
    if p_veic < 10.0:
        return False, "Pressao Abaixo do Minimo Operacional"
    return True, "OK"

def dinamica_hrs(x: np.ndarray, u: np.ndarray, dt: float = 2.0) -> np.ndarray:
    """
    Integra a dinamica termodinamica simplificada do abastecimento de H2.
    u = [taxa_rampa_bar_s, setpoint_chiller_C]
    """
    p_veic, t_veic, p_linha, t_chiller = x
    aprr, t_chill_sp = u

    # Rampa de pressao no tanque veicular limitada pela pressao da linha
    dp = aprr * dt
    p_veic_next = p_veic + dp

    # Chiller RESF-01 responde com constante de tempo de primeira ordem
    t_chiller_next = t_chiller + 0.35 * (t_chill_sp - t_chiller)

    # Aquecimento por compressao + efeito Joule-Thomson mitigado pelo chiller e troca termica
    aq_compressao = 0.085 * dp
    resfriamento = 0.045 * (t_veic - t_chiller_next) * dt
    t_veic_next = t_veic + aq_compressao - resfriamento

    # Pressao da linha acompanha a cascata (TK-BP -> TK-AP)
    p_linha_next = max(p_veic_next + 80.0, min(860.0, p_linha + 0.3 * dp))

    return np.array([p_veic_next, t_veic_next, p_linha_next, t_chiller_next], dtype=float)

### 3. Métrica Híbrida e Algoritmo *Kinodynamic RRT*

In [ ]:
SCALES = np.array([750.0, 85.0, 900.0, 60.0])

def metrica_hibrida(z_a: Tuple[np.ndarray, Tuple[int, ...]], z_b: Tuple[np.ndarray, Tuple[int, ...]], lam: float = 0.08) -> float:
    xa, qa = z_a
    xb, qb = z_b
    dist_cont = np.linalg.norm((xa - xb) / SCALES)
    dist_hamming = sum(a != b for a, b in zip(qa, qb))
    return float(dist_cont + lam * dist_hamming)

CONTROLES_ADMISSIVEIS = [
    (np.array([4.0, -40.0]), (1, 1, 1)),   # Rampa rapida + Chiller maximo (-40 C)
    (np.array([2.5, -35.0]), (1, 1, 1)),   # Rampa moderada + Chiller (-35 C)
    (np.array([1.5, -30.0]), (0, 1, 1)),   # Rampa conservadora
    (np.array([0.5, -40.0]), (0, 1, 1)),   # Rampa lenta para resfriamento termico
    (np.array([5.5, -20.0]), (1, 1, 1)),   # Rampa agressiva (tende a aquecer)
]

def executar_rrt_hrs(
    z_inicial: Tuple[np.ndarray, Tuple[int, ...]],
    p_alvo: float = 700.0,
    k_max: int = 1200,
    p_goal: float = 0.25,
    dt: float = 5.0
):
    vertices = [z_inicial]
    pais: Dict[int, Optional[int]] = {0: None}
    controles: Dict[int, Optional[np.ndarray]] = {0: None}
    descartados_sis = 0

    for k in range(k_max):
        # 1. SAMPLE com Goal Bias
        if np.random.rand() < p_goal:
            x_rand = np.array([
                np.random.uniform(700.0, 725.0),
                np.random.uniform(10.0, 75.0),
                np.random.uniform(760.0, 860.0),
                np.random.uniform(-40.0, -20.0),
            ])
            q_rand = (1, 1, 1)
        else:
            x_rand = np.array([
                np.random.uniform(40.0, 780.0),
                np.random.uniform(-20.0, 82.0),
                np.random.uniform(150.0, 880.0),
                np.random.uniform(-42.0, 25.0),
            ])
            q_rand = (1, 1, 1)
        z_rand = (x_rand, q_rand)

        # 2. NEAREST
        idx_near = min(range(len(vertices)), key=lambda idx: metrica_hibrida(vertices[idx], z_rand))
        x_near, q_near = vertices[idx_near]

        # 3. STEER (selecionar melhor controle admissivel que aproxima de z_rand e respeita SIS)
        melhor_z = None
        melhor_u = None
        melhor_dist = float("inf")

        for u_cand, q_cand in CONTROLES_ADMISSIVEIS:
            # Verificacao em subpassos (COLLISION_FREE)
            x_curr = x_near.copy()
            seguro = True
            for _ in range(5):
                x_curr = dinamica_hrs(x_curr, u_cand, dt=dt / 5.0)
                ok, _ = avaliar_sis_booleano(x_curr)
                if not ok:
                    seguro = False
                    break
            if not seguro:
                descartados_sis += 1
                continue

            d = metrica_hibrida((x_curr, q_cand), z_rand)
            if d < melhor_dist:
                melhor_dist = d
                melhor_z = (x_curr, q_cand)
                melhor_u = u_cand

        if melhor_z is None:
            continue

        # 4. EXPANDIR ARVORE
        novo_idx = len(vertices)
        vertices.append(melhor_z)
        pais[novo_idx] = idx_near
        controles[novo_idx] = melhor_u

        # 5. CRITERIO DE PARADA (Z_alvo: P_veic >= 700 bar e T_veic < 85 C)
        if melhor_z[0][0] >= p_alvo and melhor_z[0][1] < 85.0:
            caminho_idx = []
            curr = novo_idx
            while curr is not None:
                caminho_idx.append(curr)
                curr = pais[curr]
            caminho_idx.reverse()
            return vertices, pais, caminho_idx, controles, k + 1, descartados_sis

    return vertices, pais, None, controles, k_max, descartados_sis

### 4. Execução da Simulação e Análise do Grafo Discreto

In [ ]:
z_inicial = (np.array([50.0, 25.0, 200.0, 15.0]), (1, 0, 1))
t0 = time.perf_counter()
vertices, pais, caminho_idx, controles, iters, descartados = executar_rrt_hrs(z_inicial)
elapsed_ms = (time.perf_counter() - t0) * 1000

print(f"Iteracoes executadas: {iters} | Tempo: {elapsed_ms:.2f} ms")
print(f"Vertices validos na arvore: {len(vertices)} | Ramos descartados pelo SIS: {descartados}")

if caminho_idx is not None:
    traj = np.array([vertices[i][0] for i in caminho_idx])
    print(f"Trajetoria encontrada com {len(caminho_idx)} passos (~{len(caminho_idx)*5} s de abastecimento)!")
    print(f"Estado Final: P_veic={traj[-1,0]:.1f} bar | T_veic={traj[-1,1]:.1f} C | T_chiller={traj[-1,3]:.1f} C")
else:
    print("Nao foi possivel convergir no limite K_max.")

### 5. Visualização da Exploração RRT no Plano de Fase ($P_{\text{veic}} \times T_{\text{veic}}$)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Grafico 1: Arvore RRT completa no plano (P_veic, T_veic)
ax = axes[0]
for idx, p_idx in pais.items():
    if p_idx is not None:
        x1, x2 = vertices[p_idx][0], vertices[idx][0]
        ax.plot([x1[0], x2[0]], [x1[1], x2[1]], color='steelblue', alpha=0.35, linewidth=0.9)

if caminho_idx is not None:
    traj = np.array([vertices[i][0] for i in caminho_idx])
    ax.plot(traj[:, 0], traj[:, 1], color='darkgreen', linewidth=2.5, marker='o', markersize=4, label='Trajetória Segura RRT')

ax.axhline(85.0, color='red', linestyle='--', linewidth=1.5, label='Limite Térmico SIS (85 °C)')
ax.axvline(700.0, color='purple', linestyle=':', linewidth=1.5, label='Meta NWP (700 bar)')
ax.axhspan(85.0, 105.0, color='red', alpha=0.12)
ax.set_title('Árvore RRT no Plano Termodinâmico (HRS)')
ax.set_xlabel('Pressão no Tanque FCEV [bar]')
ax.set_ylabel('Temperatura no Tanque FCEV [°C]')
ax.legend(loc='upper left')

# Grafico 2: Comparacao RRT vs Rampa Direta sem Pre-resfriamento
ax2 = axes[1]
x_aberta = z_inicial[0].copy()
traj_aberta = [x_aberta.copy()]
for _ in range(len(caminho_idx) - 1 if caminho_idx else 30):
    x_aberta = dinamica_hrs(x_aberta, np.array([5.5, 20.0]), dt=5.0)
    traj_aberta.append(x_aberta.copy())
traj_aberta = np.array(traj_aberta)

tempos_rrt = np.arange(len(traj)) * 5.0
tempos_ab = np.arange(len(traj_aberta)) * 5.0
ax2.plot(tempos_rrt, traj[:, 1], color='darkgreen', linewidth=2.2, label='RRT + Chiller RESF-01')
ax2.plot(tempos_ab, traj_aberta[:, 1], color='darkorange', linewidth=2.2, linestyle='-.', label='Malha Aberta (Sem Chiller)')
ax2.axhline(85.0, color='red', linestyle='--', linewidth=1.5, label='Trip Térmico (85 °C)')
ax2.set_title('Evolução Térmica: RRT Seguro vs. Malha Aberta')
ax2.set_xlabel('Tempo de Abastecimento [s]')
ax2.set_ylabel('Temperatura FCEV [°C]')
ax2.legend()

plt.tight_layout()
plt.show()